<a href="https://colab.research.google.com/github/Dharshanthisside/ITA0616---Machine-learning/blob/main/Exp_4_Artificial_Neural_Network_with_Backpropagation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import numpy as np
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder

def sigmoid(z):
    return 1 / (1 + np.exp(-z))

def sigmoid_deriv(a):          # a = sigmoid(z)
    return a * (1 - a)

class NeuralNetwork:
    def __init__(self, n_in, n_hidden, n_out, lr=0.1, seed=42):
        rng = np.random.default_rng(seed)
        self.W1 = rng.normal(0, 0.5, (n_in, n_hidden))
        self.b1 = np.zeros((1, n_hidden))
        self.W2 = rng.normal(0, 0.5, (n_hidden, n_out))
        self.b2 = np.zeros((1, n_out))
        self.lr = lr

    def forward(self, X):
        self.a1 = sigmoid(X @ self.W1 + self.b1)
        self.a2 = sigmoid(self.a1 @ self.W2 + self.b2)
        return self.a2

    def backward(self, X, y):
        m = X.shape[0]
        d2 = (self.a2 - y) * sigmoid_deriv(self.a2)          # output error
        d1 = (d2 @ self.W2.T) * sigmoid_deriv(self.a1)       # hidden error
        self.W2 -= self.lr * self.a1.T @ d2 / m
        self.b2 -= self.lr * d2.sum(axis=0, keepdims=True) / m
        self.W1 -= self.lr * X.T @ d1 / m
        self.b1 -= self.lr * d1.sum(axis=0, keepdims=True) / m

    def train(self, X, y, epochs=5000, verbose=1000):
        for e in range(1, epochs + 1):
            out = self.forward(X)
            self.backward(X, y)
            if e % verbose == 0:
                print(f"Epoch {e:5d}  MSE = {np.mean((y - out) ** 2):.5f}")

    def predict(self, X):
        return self.forward(X)

# ---- Test 1: XOR ----
print("=== XOR problem ===")
X_xor = np.array([[0,0],[0,1],[1,0],[1,1]])
y_xor = np.array([[0],[1],[1],[0]])
nn = NeuralNetwork(2, 4, 1, lr=1.0)
nn.train(X_xor, y_xor, epochs=10000, verbose=2500)
print("Predictions:", np.round(nn.predict(X_xor).ravel(), 3))

# ---- Test 2: Iris ----
print("\n=== Iris dataset ===")
iris = load_iris()
X = StandardScaler().fit_transform(iris.data)
y = OneHotEncoder(sparse_output=False).fit_transform(iris.target.reshape(-1, 1))
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.2, random_state=1)

nn = NeuralNetwork(4, 8, 3, lr=0.5)
nn.train(X_tr, y_tr, epochs=5000, verbose=1000)

pred = np.argmax(nn.predict(X_te), axis=1)
actual = np.argmax(y_te, axis=1)
print("Test accuracy:", np.mean(pred == actual))

=== XOR problem ===
Epoch  2500  MSE = 0.01036
Epoch  5000  MSE = 0.00205
Epoch  7500  MSE = 0.00108
Epoch 10000  MSE = 0.00072
Predictions: [0.023 0.975 0.974 0.032]

=== Iris dataset ===
Epoch  1000  MSE = 0.03836
Epoch  2000  MSE = 0.01726
Epoch  3000  MSE = 0.01353
Epoch  4000  MSE = 0.01217
Epoch  5000  MSE = 0.01146
Test accuracy: 0.9666666666666667
